# DSERT-RoLL encoder analysis: t-SNE, probes, BEV Grad-CAM

**Before running:** Settings → Accelerator **GPU T4**, Internet **On**, and *Add Input* → your
`dsert-roll-analysis-bundle` dataset (made with `make_kaggle_bundle.sh` on Snellius).

Edit only the next cell to choose which models to compare.

In [ ]:
REPO_URL = "https://github.com/MuhammadIbneRafiq/DSERT-RoLL-3DOD.git"

# name -> (checkpoint file in the bundle's checkpoints/ folder, sensor fed to the model)
MODELS = {
    "radar_scratch":  ("radar_scratch.pth",  "radar"),
    "livox_pretrain": ("livox_pretrain.pth", "livox"),
    # "radar_ft_from_livox": ("radar_ft_from_livox.pth", "radar"),   # run 3, once it exists
    # "livox_enc_on_radar":  ("livox_pretrain.pth", "radar"),         # LiDAR-trained model fed radar, no fine-tuning
}
MAX_FRAMES = 300          # validation frames per model for features
GRADCAM_WEATHER = "Fog"   # Clear | Fog | Heavy_Rain | Light_Rain | Heavy_Snow | Light_Snow | None (any)
GRADCAM_FRAMES = 4

In [ ]:
# 1) Code + environment (~5-10 min the first time in a session)
import os
REPO = "/kaggle/working/DSERT-RoLL-3DOD"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
!cd {REPO} && git pull -q && git log --oneline -1
!bash {REPO}/detection/tools/scripts/kaggle/kaggle_setup.sh

In [ ]:
# 2) Find the uploaded bundle (Kaggle may keep the folders zipped)
import glob, zipfile
def find_bundle_dir(marker, zip_name):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)]
    if not hits:
        for z in glob.glob(f"/kaggle/input/**/{zip_name}", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/bundle")
        hits = glob.glob(f"/kaggle/working/bundle/**/{marker}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{marker} not found: is the dataset added as input?")
    return os.path.dirname(sorted(hits, key=len)[0])

DATA_ROOT = find_bundle_dir("ImageSets", "data.zip")
CKPT_DIR = find_bundle_dir("*.pth", "checkpoints.zip")
print("data:", DATA_ROOT, "| sequences:", len(open(f"{DATA_ROOT}/ImageSets/final/val.txt").read().split()))
print("checkpoints:", CKPT_DIR, sorted(os.listdir(CKPT_DIR)))
TOOLS = f"{REPO}/detection/tools"
OUT = "/kaggle/working/analysis"
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 3) Encoder features: one vector per labelled object (+ class, weather, distance, points, detected?)
for name, (ckpt, sensor) in MODELS.items():
    print(f"===== {name} ({sensor}) =====")
    !python {TOOLS}/analysis/extract_features.py --ckpt {CKPT_DIR}/{ckpt} --sensor {sensor} --data_root {DATA_ROOT} --name {name} --out {OUT}/feats_{name}.npz --max_frames {MAX_FRAMES} 2>&1 | grep -v Warning

In [ ]:
# 4) Compare: metrics, recall tables, t-SNE
from IPython.display import Image, display
feats = " ".join(f"{OUT}/feats_{n}.npz" for n in MODELS)
!python {TOOLS}/analysis/plot_tsne.py --features {feats} --out_dir {OUT}
display(Image(f"{OUT}/recall_comparison.png"))
for n in MODELS:
    display(Image(f"{OUT}/tsne_{n}.png"))

In [ ]:
# 5) BEV Grad-CAM: where each model looks for a few objects (missed ones first)
weather_arg = f"--weather {GRADCAM_WEATHER}" if GRADCAM_WEATHER else ""
for name, (ckpt, sensor) in MODELS.items():
    !python {TOOLS}/analysis/gradcam_bev.py --ckpt {CKPT_DIR}/{ckpt} --sensor {sensor} --data_root {DATA_ROOT} --name {name} --out_dir {OUT}/gradcam_{name} --max_frames {GRADCAM_FRAMES} {weather_arg} 2>&1 | grep -E "wrote|Error|error"
for name in MODELS:
    for png in sorted(glob.glob(f"{OUT}/gradcam_{name}/*.png"))[:GRADCAM_FRAMES]:
        print(png); display(Image(png))

## Reading the results

**Metrics table** (comparable across models; t-SNE pictures are not):
- `knn_class_acc`, `silhouette_class` higher = classes better separated in the encoder's features.
- `knn_weather_acc` higher = an object's features change more with the weather (less weather-robust).
- `recall_all` = fraction of labelled objects the full model detected.

**Recall tables / bar chart** show *where* a model breaks: by weather, by distance, by points inside the box.
For pretraining, compare `radar_ft_from_livox` vs `radar_scratch` (same radar input): gains at long range or with
few points suggest the LiDAR-pretrained encoder helps where radar evidence is thin.

**t-SNE**: tight, separated class clusters = good. Missed objects (red) clustering together = a systematic failure
(e.g. far, few points, or one weather). Clusters split by weather = weather-dependent features.

**Grad-CAM**: bright on and around the object = the score comes from the object itself; bright on unrelated
clutter or empty space = the model relies on context or is being misled (common for missed objects in bad weather).

`livox_pretrain` is fed LiDAR, the radar models are fed radar, so compare their numbers as context, not as a
head-to-head: the clean comparison is between models given the same sensor.